# Friedel Pair Matching – batch processing

`ImageD11.match_friedel_pairs.py` is a standalone pipeline script for Friedel pair matching in scanning_3DXRD datasets. 
 
Usage — run locally:
-------
    python match_friedel_pairs.py -dsfile /path/to/dataset.h5 \
                                  [-parfile /path/to/params.json] \
                                  [-pairing_options /path/to/options.json] \
                                  [-use2Dpeaks True]
 
Usage — submit to slurm:
-------
    python match_friedel_pairs.py -dsfile /path/to/dataset.h5 \
                                  [-parfile /path/to/params.json] \
                                  [-pairing_options /path/to/options.json] \
                                  [-use2Dpeaks True]
                                  -usecluster True
 
Pairing options JSON keys (all optional — defaults shown):
-------
    # tolerances
    tol_gv             : 0.05
    tol_eta            : 0.2
    tol_logI           : null          (null -> np.inf)
    weights            : {"gx":1,"gy":1,"gz":1,"eta":1,"I":1}
    # pairing strategy
    pair_type          : "omega"       ("omega" | "eta" | "all")
    filter_mode        : "relaxed"
    n_eta_bins         : 360
    drop_unpaired      : false
    extended_bin_search: false
    n_workers          : -1            -1 -> all availables
    timeout            : 120
    n_steps            : 25
    # slurm / cluster options
    slurm_partition    : "nice"
    slurm_mem_G        : 64
    slurm_time         : "02:00:00"
    slurm_cpus         : 16
    n_steps           : 30

In [ ]:
import os, sys, time, glob
start = time.time()

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

In [ ]:
from ImageD11 import match_friedel_pairs
import ImageD11.sinograms.dataset
import subprocess

%load_ext autoreload
%autoreload 2

#### build a dictionnary of samples and datasets to process

In [ ]:
# check samples and datasets to process
dataroot = '/data/visitor/es1832/id11/20260715/PROCESSED_DATA/'  # root directory of your project

# samples list: guess it from folders in dataroot or define manually

#skip = ['ipynb','NOTEBOOKS','pars','cif', 'Si_cube', '__pycache__','json']   # skip any folder that contains any of these in its name
#samples = [p for p in os.listdir(dataroot) if all([sk not in p for sk in skip])]  # use it if you want to run all samples

samples = ['BER_01']   # manual sample list

In [ ]:
# find datasets to process for each sample
skip = ['ipynb', 'match','pct', 'pycache']  # skip list for datasets names

process_dict = {}

for s in samples:
    sample_dir = os.path.join(dataroot,s)
    dsets = [ds for ds in os.listdir(sample_dir) if all([sk not in ds for sk in skip])]
    process_dict[s] = dsets

# check process_dict: should look like this : {sample_1:[dset1, dset2,...], sample_2:[dset1, dset2,...],...}
# if only one dataset associated witha  sample, shoudl be in a list of one element: sample1:[dset1]
process_dict

In [ ]:
# have a look at default options in find_friedel_pairs.py 
OPTS = match_friedel_pairs.Options()
print(OPTS)

In [ ]:
# update options according to the best guess in fp1 notebook, then save to a json file; 
OPTS.ds_max = 1.6
OPTS.tol_gv = 0.05
OPTS.tol_eta = 0.3
OPTS.tol_logI = 0.5
OPTS.weights = {'I':0.5}
OPTS.n_eta_bins = 360
OPTS.pair_type = 'omega'
OPTS.drop_unpaired = False
OPTS.extended_bin_search = False
OPTS.slurm_cpus = 24
OPTS.slurm_mem_G = 200
OPTS.y0 = 11.22333333333333

In [ ]:
pairing_pars = os.path.join(dataroot,samples[0],'friedel_pairs_opts.json')
OPTS.save(pairing_pars)

In [ ]:
# build command queue
def get_dsfile(sample, dsname):
    return os.path.join(dataroot, sample, dsname, f'{dsname}_dataset.h5')

# define here if you want to use the cluster (slurm). recommended
def get_command(sample, dsname):
    return f'{match_friedel_pairs.__file__} -dsfile {get_dsfile(sample,dsname)} -pairing_options {pairing_pars} -use2Dpeaks True -usecluster True' 
    

command_queue = []
for sample, dset_list in process_dict.items():
    for dsname in dset_list:
        command_queue.append(get_command(sample, dsname))

# check it looks correct
command_queue[0]               

In [ ]:
# submit jobs
for command in command_queue:
    !python {command}

In [ ]:
# check job is running in slurm
!squeue --me